# 波動伝搬：フレネル回折とフラウンホーファー回折

![Fig. 8](../.figure/fig8.jpg)

## 1. 座標系と基本概念

波動の自由空間伝搬を考える。開口面（aperture plane）を $z=0$ 平面とし、その上の座標を

$$
(\xi,\eta)
$$

とする。観測面（observation plane）は開口面から距離 $z$ だけ離れた平面であり、その上の座標を

$$
(x,y)
$$

とおく。

開口面直後の複素振幅を

$$
u_0(\xi,\eta)
$$

とし、距離 $z$ だけ伝搬した後の点 $P$ における複素振幅を

$$
u_P(x,y;z)
$$

と書く。

スカラーで表した波数は $k=\frac{2\pi}{\lambda}$ である。ここで $\lambda$ は波長である。

開口面上の一点 $(\xi,\eta,0)$ から観測点 $P(x,y,z)$ までの距離は

$$
r=\sqrt{z^2+(x-\xi)^2+(y-\eta)^2}
$$

である。

ホイヘンスの原理によれば、開口面上の各点は二次球面波を放出する点光源のように振る舞う。したがって、観測面の場は開口面上のすべての点から出た球面波の重ね合わせとして与えられる。

ただし、次のように区別して理解するのが正確である。

- 近距離では、各点光源から出た球面波の位相差をそのまま考慮しなければならない。この領域がフレネル回折領域である。
- 十分に遠距離では、点光源がつくる波面を平面波とみなし、各点光源が形成する平面波の位相変化は観測方向ごとにほぼ線形に整理される。この領域がフラウンホーファー回折領域である。

---



In [ ]:
# 第1節：数値グリッド、開口場、物理定数
import numpy as np
import matplotlib.pyplot as plt

wavelength = 633e-9          # 波長 [m]
k = 2 * np.pi / wavelength   # 波数 [rad/m]
N = 512                      # 各次元のグリッド点数
L = 5.0e-3                   # 開口面ウィンドウの一辺の長さ [m]
dx = L / N                   # 開口面でのサンプリング間隔 [m]

aperture_axis = (np.arange(N) - N // 2) * dx
XI, ETA = np.meshgrid(aperture_axis, aperture_axis, indexing="xy")

# 入力場の例：弱いガウス型位相物体を持つ正方形開口。
aperture_width = 0.2e-3
u0 = ((np.abs(XI) <= aperture_width / 2) & (np.abs(ETA) <= aperture_width / 2)).astype(complex)
phase_object = np.exp(1j * 0.8 * np.exp(-(XI**2 + ETA**2) / (0.18e-3)**2))
u0 *= phase_object

print(f"グリッド: {N} x {N}, dx = {dx:.3e} m")
print(f"波長 = {wavelength:.3e} m")
print(f"開口内の非ゼロサンプル数 = {np.count_nonzero(np.abs(u0) > 0)}")

fig, ax = plt.subplots(figsize=(4, 3.5))
im = ax.imshow(
    np.abs(u0),
    extent=[aperture_axis[0] * 1e3, aperture_axis[-1] * 1e3, aperture_axis[0] * 1e3, aperture_axis[-1] * 1e3],
    cmap="gray",
)
ax.set_xlabel("x [mm]")
ax.set_ylabel("y [mm]")
ax.set_title("Aperture amplitude |u0|")
fig.colorbar(im, ax=ax, label="amplitude")
fig.tight_layout()


## 2. ホイヘンス・キルヒホッフ積分

開口面より前で波がどのように形成されたかは考えず、開口面直後の複素振幅 $u_0(\xi,\eta)$ が与えられているとする。

ホイヘンス・キルヒホッフ回折積分の一つの形は次のように書ける。

$$
u(x,y;z)
=
-\frac{ik}{4\pi}
\iint
u_0(\xi,\eta)
\frac{e^{ikr}}{r}
\left(1+\cos\theta\right)
\,d\xi d\eta .
$$

ここで $\theta$ は、開口面の法線方向と観測方向がなす角である。また、

$$
k=\frac{2\pi}{\lambda}
$$

なので、上式は

$$
u(x,y;z)
=
-\frac{i}{2\lambda}
\iint
u_0(\xi,\eta)
\frac{e^{ikr}}{r}
\left(1+\cos\theta\right)
\,d\xi d\eta
$$

と書ける。

傾斜因子（obliquity factor）を

$$
K(\theta)=\frac{1+\cos\theta}{2}
$$

とおくと、

$$
u(x,y;z)
=
-\frac{i}{\lambda}
\iint
u_0(\xi,\eta)
\frac{e^{ikr}}{r}
K(\theta)
\,d\xi d\eta .
$$

近軸近似、すなわち観測角 $\theta$ が十分に小さい場合には

$$
\cos\theta \approx 1
$$

であるため、

$$
K(\theta)\approx 1
$$

となる。したがって、

$$
u(x,y;z)
\approx
-\frac{i}{\lambda}
\iint
u_0(\xi,\eta)
\frac{e^{ikr}}{r}
\,d\xi d\eta .
$$

また、振幅項に含まれる $r$ は位相項 $e^{ikr}$ に比べてゆっくり変化するので、

$$
r \approx z
$$

とおくことができる。すると、

$$
u(x,y;z)
\approx
-\frac{i}{\lambda z}
\iint
u_0(\xi,\eta)
e^{ikr}
\,d\xi d\eta .
$$

ここで重要な点は次のとおりである。

振幅に対応する分母の $r$ は $z$ で近似できるが、指数関数の中にある $r$ を不用意に $z$ に置き換えてはいけない。光学では $k=2\pi/\lambda$ が非常に大きいため、ごく小さな距離差でも大きな位相差を生み得るからである。

---

## 3. 距離 $r$ のテイラー展開

ここで、指数関数の中にある

$$
r=\sqrt{z^2+(x-\xi)^2+(y-\eta)^2}
$$

を近似する。

まず、

$$
r
=
z
\sqrt{
1+
\frac{(x-\xi)^2+(y-\eta)^2}{z^2}
}
$$

と書ける。

ここで

$$
\sqrt{1+s}
=
1+\frac{s}{2}-\frac{s^2}{8}+\cdots
$$

を用いると、

$$
r
=
z
\left[
1
+
\frac{1}{2}
\frac{(x-\xi)^2+(y-\eta)^2}{z^2}
-
\frac{1}{8}
\left(
\frac{(x-\xi)^2+(y-\eta)^2}{z^2}
\right)^2
+
\cdots
\right].
$$

したがって、

$$
r
\approx
z
+
\frac{(x-\xi)^2+(y-\eta)^2}{2z}
$$

となる。

注意すべき点は、テイラー展開の後続項が小さく見えるからといって、ただちに捨てられるわけではないことである。実際に捨てられる項が生む位相誤差

$$
k\Delta r
$$

が十分に小さい場合に、フレネル近似が成立する。

この近似をホイヘンス・キルヒホッフ積分に代入すると、

$$
e^{ikr}
\approx
e^{ikz}
\exp
\left[
i\frac{k}{2z}
\left\{
(x-\xi)^2+(y-\eta)^2
\right\}
\right].
$$

したがって、

$$
u(x,y;z)
\approx
-\frac{i e^{ikz}}{\lambda z}
\iint
u_0(\xi,\eta)
\exp
\left[
i\frac{k}{2z}
\left\{
(x-\xi)^2+(y-\eta)^2
\right\}
\right]
\,d\xi d\eta .
$$

$k=2\pi/\lambda$ なので、

$$
\frac{k}{2z}
=
\frac{\pi}{\lambda z}
$$

である。したがって、フレネル回折積分は

$$
u(x,y;z)
\approx
-\frac{i e^{ikz}}{\lambda z}
\iint
u_0(\xi,\eta)
\exp
\left[
i\frac{\pi}{\lambda z}
\left\{
(x-\xi)^2+(y-\eta)^2
\right\}
\right]
\,d\xi d\eta .
$$

また、

$$
\frac{1}{i}=-i
$$

を用いて、同じ式を

$$
u(x,y;z)
\approx
\frac{e^{ikz}}{i\lambda z}
\iint
u_0(\xi,\eta)
\exp
\left[
i\frac{\pi}{\lambda z}
\left\{
(x-\xi)^2+(y-\eta)^2
\right\}
\right]
\,d\xi d\eta
$$

と書いてもよい。

---

## 4. フレネル回折積分のフーリエ変換形式

次に、指数項の中の二乗を展開する。

$$
(x-\xi)^2+(y-\eta)^2
=
x^2+y^2+\xi^2+\eta^2-2x\xi-2y\eta .
$$

したがって、

$$
u(x,y;z)
\approx
\frac{e^{ikz}}{i\lambda z}
\iint
u_0(\xi,\eta)
\exp
\left[
i\frac{\pi}{\lambda z}
\left(
x^2+y^2+\xi^2+\eta^2-2x\xi-2y\eta
\right)
\right]
\,d\xi d\eta .
$$

ここで、$x$ と $y$ だけを含む項は積分変数 $\xi,\eta$ に依存しないため、積分の外に出すことができる。

$$
u(x,y;z)
\approx
\frac{e^{ikz}}{i\lambda z}
\exp
\left[
i\frac{\pi}{\lambda z}
(x^2+y^2)
\right]
\iint
u_0(\xi,\eta)
\exp
\left[
i\frac{\pi}{\lambda z}
(\xi^2+\eta^2)
\right]
\exp
\left[
-i\frac{2\pi}{\lambda z}
(x\xi+y\eta)
\right]
\,d\xi d\eta .
$$

この式は、フレネル回折積分の重要な形である。

任意の関数 $g(\xi,\eta)$ のフーリエ変換を次のように定義する。

$$
\mathcal{F}\{g(\xi,\eta)\}(f_x,f_y)
=
\iint
g(\xi,\eta)
e^{-i2\pi(f_x\xi+f_y\eta)}
\,d\xi d\eta .
$$

すると、フレネル回折は

$$
g(\xi,\eta)
=
u_0(\xi,\eta)
\exp
\left[
i\frac{\pi}{\lambda z}
(\xi^2+\eta^2)
\right]
$$

のフーリエ変換とみなすことができる。このとき、フーリエ変換の空間周波数は

$$
f_x=\frac{x}{\lambda z},
\qquad
f_y=\frac{y}{\lambda z}
$$

である。

したがって、フレネル回折は単純なフーリエ変換ではなく、

$$
u_0(\xi,\eta)
$$

に二次位相項（chirp phase）

$$
\exp
\left[
i\frac{\pi}{\lambda z}
(\xi^2+\eta^2)
\right]
$$

を掛けてからフーリエ変換した形である。

---



In [ ]:
# 第4節：フレネル回折のフーリエ変換形式
# このセルには、フレネルのフーリエ形式による伝搬に必要な補助関数をまとめている。
def fft2c(field):
    """表示用配列に合わせた光学の慣例による中心化2次元FFT。"""
    return np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(field)))

def fresnel_fourier_form(u0, dx, z, wavelength):
    """フレネル回折積分をフーリエ変換形式で評価する。

    サンプリングされたフーリエ周波数 (fx, fy) は、x = wavelength * z * fx および
    y = wavelength * z * fy によって物理的な観測座標へ変換される。
    """
    n_y, n_x = u0.shape
    if n_x != n_y:
        raise ValueError("この教材例では正方形の入力配列を想定しています。")

    aperture_axis = (np.arange(n_x) - n_x // 2) * dx
    XI, ETA = np.meshgrid(aperture_axis, aperture_axis, indexing="xy")

    frequencies = np.fft.fftshift(np.fft.fftfreq(n_x, d=dx))
    observation_axis = wavelength * z * frequencies
    X, Y = np.meshgrid(observation_axis, observation_axis, indexing="xy")

    k = 2 * np.pi / wavelength
    input_chirp = np.exp(1j * np.pi * (XI**2 + ETA**2) / (wavelength * z))
    output_chirp = np.exp(1j * np.pi * (X**2 + Y**2) / (wavelength * z))
    prefactor = np.exp(1j * k * z) / (1j * wavelength * z)

    spectrum = fft2c(u0 * input_chirp) * dx**2
    propagated_field = prefactor * output_chirp * spectrum
    return propagated_field, observation_axis

z_fresnel = 0.25
u_fresnel, x_fresnel = fresnel_fourier_form(u0, dx, z_fresnel, wavelength)
I_fresnel = np.abs(u_fresnel)**2
I_fresnel /= I_fresnel.max()

print(f"フレネル伝搬距離 z = {z_fresnel:.3f} m")
print(f"フレネル観測ウィンドウ = {x_fresnel[0]*1e3:.2f} から {x_fresnel[-1]*1e3:.2f} mm")
print(f"フレネル正規化強度: min = {I_fresnel.min():.3e}, max = {I_fresnel.max():.3f}")
print("フレネル中心線サンプル:", np.array2string(I_fresnel[N // 2, N // 2 - 3:N // 2 + 4], precision=3))

fig, axes = plt.subplots(1, 2, figsize=(8, 3.4))
im = axes[0].imshow(
    I_fresnel,
    extent=[x_fresnel[0] * 1e3, x_fresnel[-1] * 1e3, x_fresnel[0] * 1e3, x_fresnel[-1] * 1e3],
    cmap="magma",
)
axes[0].set_xlabel("x [mm]")
axes[0].set_ylabel("y [mm]")
axes[0].set_title(f"Fresnel intensity, z={z_fresnel:.2f} m")
fig.colorbar(im, ax=axes[0], label="normalized intensity")

center = I_fresnel.shape[0] // 2
axes[1].plot(x_fresnel * 1e3, I_fresnel[center])
axes[1].set_xlim(-5, 5)
axes[1].set_xlabel("x [mm]")
axes[1].set_ylabel("normalized intensity")
axes[1].set_title("center line")
axes[1].grid(True)
fig.tight_layout()


## 5. フラウンホーファー回折

フラウンホーファー回折は、フレネル回折をさらに一段階近似したものである。

フレネル積分の中には、次のような開口面側の二次位相項がある。

$$
\exp
\left[
i\frac{\pi}{\lambda z}
(\xi^2+\eta^2)
\right].
$$

フラウンホーファー近似では、この項の位相変化が開口面全体で十分に小さいと仮定する。

すなわち、

$$
\frac{\pi}{\lambda z}(\xi^2+\eta^2) \ll 1
$$

であれば、

$$
\exp
\left[
i\frac{\pi}{\lambda z}
(\xi^2+\eta^2)
\right]
\approx 1
$$

とおける。

開口の代表的な大きさを $D$ とすると、おおよそ

$$
z \gg \frac{D^2}{\lambda}
$$

のとき、フラウンホーファー近似はよく成立する。

この近似を適用すると、

$$
u_F(x,y;z)
\approx
\frac{e^{ikz}}{i\lambda z}
\exp
\left[
i\frac{\pi}{\lambda z}
(x^2+y^2)
\right]
\iint
u_0(\xi,\eta)
\exp
\left[
-i\frac{2\pi}{\lambda z}
(x\xi+y\eta)
\right]
\,d\xi d\eta .
$$

フーリエ変換の定義と比較すると、

$$
u_F(x,y;z)
\approx
\frac{e^{ikz}}{i\lambda z}
\exp
\left[
i\frac{\pi}{\lambda z}
(x^2+y^2)
\right]
\mathcal{F}\{u_0(\xi,\eta)\}
\left(
\frac{x}{\lambda z},
\frac{y}{\lambda z}
\right).
$$

つまり、フラウンホーファー回折において観測面の複素振幅は、開口面の複素振幅のフーリエ変換に比例する。

観測される強度は

$$
I_F(x,y;z)
=
|u_F(x,y;z)|^2
$$

であるため、前に掛かっている純粋な位相項は強度に影響しない。したがって、フラウンホーファー領域の強度パターンは

$$
I_F(x,y;z)
\propto
\left|
\mathcal{F}\{u_0(\xi,\eta)\}
\left(
\frac{x}{\lambda z},
\frac{y}{\lambda z}
\right)
\right|^2
$$

で与えられる。

ただし、この関係におけるフーリエ変換の座標は物理的な観測座標 $(x,y)$ ではなく、

$$
\left(
\frac{x}{\lambda z},
\frac{y}{\lambda z}
\right)
$$

にスケールされた空間周波数座標である。


In [ ]:
# 第5節：フラウンホーファー回折
# フラウンホーファー伝搬では、第4節の中心化FFT補助関数を再利用する。
def fraunhofer_propagate(u0, dx, z, wavelength):
    """開口場のフラウンホーファー回折パターンを評価する。"""
    n_y, n_x = u0.shape
    if n_x != n_y:
        raise ValueError("この教材例では正方形の入力配列を想定しています。")

    frequencies = np.fft.fftshift(np.fft.fftfreq(n_x, d=dx))
    observation_axis = wavelength * z * frequencies
    X, Y = np.meshgrid(observation_axis, observation_axis, indexing="xy")

    k = 2 * np.pi / wavelength
    aperture_transform = fft2c(u0) * dx**2
    prefactor = np.exp(1j * k * z) / (1j * wavelength * z)
    output_chirp = np.exp(1j * np.pi * (X**2 + Y**2) / (wavelength * z))

    propagated_field = prefactor * output_chirp * aperture_transform
    return propagated_field, observation_axis

z_far = 5.0
u_fraunhofer, x_far = fraunhofer_propagate(u0, dx, z_far, wavelength)
I_fraunhofer = np.abs(u_fraunhofer)**2
I_fraunhofer /= I_fraunhofer.max()

fraunhofer_distance = aperture_width**2 / wavelength
print(f"D^2/lambda = {fraunhofer_distance:.3f} m; 選択した z = {z_far:.3f} m")
print(f"フラウンホーファー観測ウィンドウ = {x_far[0]*1e3:.2f} から {x_far[-1]*1e3:.2f} mm")
print(f"フラウンホーファー正規化強度: min = {I_fraunhofer.min():.3e}, max = {I_fraunhofer.max():.3f}")
print("フラウンホーファー中心線サンプル:", np.array2string(I_fraunhofer[N // 2, N // 2 - 3:N // 2 + 4], precision=3))

fig, axes = plt.subplots(1, 2, figsize=(8, 3.4))
im = axes[0].imshow(
    I_fraunhofer,
    extent=[x_far[0] * 1e3, x_far[-1] * 1e3, x_far[0] * 1e3, x_far[-1] * 1e3],
    cmap="magma",
)
axes[0].set_xlabel("x [mm]")
axes[0].set_ylabel("y [mm]")
axes[0].set_title(f"Fraunhofer intensity, z={z_far:.1f} m")
fig.colorbar(im, ax=axes[0], label="normalized intensity")

center = I_fraunhofer.shape[0] // 2
axes[1].plot(x_far * 1e3, I_fraunhofer[center])
axes[1].set_xlim(-20, 20)
axes[1].set_xlabel("x [mm]")
axes[1].set_ylabel("normalized intensity")
axes[1].set_title("center line")
axes[1].grid(True)
fig.tight_layout()
